<a href="https://colab.research.google.com/github/Sulamithsingh/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method Choice

I selected a Random Forest Classifier for this task because it can learn nonlinear relationships between features and provides feature importance scores for interpretation. Compared with the Week 4 rule-based baseline, the model can automatically learn patterns from the data instead of relying on fixed thresholds.

In [21]:
import os

print(os.getcwd())

/content/flyrank-ml-internship-starter/work/notebooks


In [22]:
import pandas as pd

df = pd.read_csv("/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [23]:
df["target"] = (
    (df["days_since_last_update"] >= 90) &
    (df["impressions_90d"] >= 1000) &
    (df["ctr"] <= 0.10) &
    (df["avg_position"] <= 20)
).astype(int)

features = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position"
]

X = df[features]
y = df["target"]

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Split design

I used an 80/20 train-test split with a fixed random state of 42. This provides a fair evaluation by training the model on most of the data while testing it on unseen examples.

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(24000, 4)
(6000, 4)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## Train + compare vs my baseline

The Week 4 baseline used manually defined rules to identify pages that should be refreshed. In this notebook, I trained a Random Forest classifier using the same input features. The model is then evaluated on the test set and compared with the rule-based baseline.

In [25]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

accuracy = accuracy_score(y_test, pred)

print("Accuracy:", accuracy)

Accuracy: 1.0


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Errors and interpretation

The Random Forest model uses multiple decision trees to learn patterns from the dataset. Compared with the Week 4 rule-based baseline, it can capture more complex relationships between features instead of relying on fixed thresholds.

The most influential features include:
- Days since last update
- Impressions in the last 90 days
- Click-through rate (CTR)
- Average search position

Some prediction errors may occur for pages with feature values close to the decision thresholds or for pages that have unusual traffic patterns. Overall, the model provides a more flexible approach than the rule-based baseline.

In [26]:
importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

print("Classification Report")
print(classification_report(y_test, pred))

print("\nFeature Importance")
display(importance.sort_values(by="Importance", ascending=False))

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5809
           1       1.00      1.00      1.00       191

    accuracy                           1.00      6000
   macro avg       1.00      1.00      1.00      6000
weighted avg       1.00      1.00      1.00      6000


Feature Importance


,Feature,Importance
3,avg_position,0.331056
2,ctr,0.303488
0,days_since_last_update,0.212908
1,impressions_90d,0.152548


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.